# Latest Page Scraper

- Started off scraping the album reviews from [rateyourmusic.com/latest](http://rateyourmusic.com/latest).
- The web scraper scrolled through the most recent user reviews for any given album and the pulled all the albums name, artist name, user review and user rating.
- This gave us ~2,000 reviews fairly quickly. However, this section of the website limits us to only 2,000 reviews, so we had to go deeper into the website to get the full 8,000.


### Details
- This scraper Opened Chrome with **undetected_chromedriver.**
- Detected if Cloudflare showed a “human check”, paused if so to allow to solve.
- Browse through the **Latest Reviews** pages, collect the **album, artist, review text and user rating.**
- Save everything to CSV.

In [ ]:
# Imports and Setup
import time, random, re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

# Configuration 
BASE = "https://rateyourmusic.com"           # Set base website as the root
LATEST_URL = f"{BASE}/latest"                # The page that we are actually scraping
OUT_FILE = "rym_latest_reviews_shared.csv"   # CSV we are saving to
PAGES_TO_SCRAPE = 50                         
HUMAN_WAIT_SECS = 420                        # Time to wait to solve the human verifcation check 
DELAY = (1.0, 2.0)                           # Delay between pages to avoid clouflare detection


# Used to find things like "4.5 stars" in HTML
STAR_RE = re.compile(r"([\d.]+)\s*star", re.I)


# Driver
def make_driver():
    """ Launches Chrome with special option to hide signals that usually give away selenium """
    opts = uc.ChromeOptions()
    opts.add_argument("--disable-blink-features=AutomationControlled")
    return uc.Chrome(options=opts)

# Cloudflare Detection Handling
def is_human_check(driver):
    """ Looks at the Page and nnotices if it finds or has hit any verification alert """
    src = driver.page_source.lower()
    cues = ["confirm you are human","verify you are human","i'm not a robot","cf-challenge"]
    if any(c in src for c in cues):
        return True
    try:
        return bool(driver.find_elements(By.CSS_SELECTOR, "iframe[src*='recaptcha'], #challenge-stage"))
    except:
        return False

def wait_for_clear(driver, timeout=HUMAN_WAIT_SECS):
    """ If the human check appears, program will wait while we solve in Chrome. Checks every 2 seconds if challenege is gone """
    start = time.time()
    while time.time() - start < timeout:
        if not is_human_check(driver):
            return True
        time.sleep(2)
    return False

def load_page(driver, url):
    """ Opens a URL page in Chrome and if Cloudflare shows up, it pauses and allows us to solve. 
        Then returns the page after parting with BS """
    driver.get(url)
    if is_human_check(driver):
        if not wait_for_clear(driver):
            return None
    time.sleep(1)
    return BeautifulSoup(driver.page_source, "lxml")

# Extracting Data
def extract_review(block):
    """ Pulling Review Text and Ratings (looking for stars) """
    text = " ".join(el.get_text(" ", strip=True) for el in block.select("span.rendered_text, p")).strip()
    rating = ""
    tag = block.select_one("[aria-label*='star'], [title*='star']")
    if tag:
        m = STAR_RE.search(tag.get("aria-label","")+tag.get("title",""))
        if m: rating = m.group(1)
    return text, rating

def extract_album_artist(block):
    """ Pulls the album name and the artist name """
    album, artist = "", ""
    rel = block.select_one("a[href*='/release/']")
    if rel: album = rel.get_text(" ", strip=True)
    art = block.select_one("a[href*='/artist/']")
    if art: artist = art.get_text(" ", strip=True)
    return album, artist

# Main Scraper Loop
def run():
    """ The scraper:
        - Loops through the pages of the Latest Reviews
        - Finds all review blocks and pulls the album, artist, review text and rating.
        - Appends them to a list 
        - Saves the list to a CSV
    """
    driver = make_driver()
    all_reviews = []
    try:
        for page in range(1, PAGES_TO_SCRAPE + 1):
            soup = load_page(driver, f"{LATEST_URL}?page={page}")
            if not soup: break
            blocks = soup.select("div.review, div.review_box, table.mbgen")
            if not blocks: break
            for block in blocks:
                album, artist = extract_album_artist(block)
                text, rating = extract_review(block)
                if text or rating:
                    all_reviews.append({
                        "album": album,
                        "artist": artist,
                        "review_text": text,
                        "user_rating": rating
                    })
            pd.DataFrame(all_reviews).to_csv(OUT_FILE, index=False)
            time.sleep(random.uniform(*DELAY))
    finally:
        driver.quit()

if __name__ == "__main__":
    run()


# Charts Scraper
- In search of more album reviews, we switched to seeding albums from (Top albums of all time, 2024 and 2025 top albums)
- For each album found in those charts, the scraper went into its reviews page and collected reviews for each album, allowing us to get a larger variety of genres and artists and pull the require data amount.


### Details

**Album Collection**
- Opens the chart pages (all time, 2024 and 2025) and collects links to album pages


**Cloudflare Handling**
- If a "confirm you are human" page appears, the script pauses and allows us to solve the challenege in Chrome
- Once solve, the program continues automatically.

**Scraping Reviews**
- Opens the album's review page and collects up to set "Reviews Per Album"
- It scrapes the album name, the artist name, the review text and the user rating.

In [ ]:
# Imports
import time, random, re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By


# Config
BASE = "https://rateyourmusic.com"
CHARTS = [
    f"{BASE}/charts/top/album/all-time",
    f"{BASE}/charts/top/album/2025",
    f"{BASE}/charts/top/album/2024"
]
OUT_FILE = "rym_chart_reviews.csv"
REVIEWS_PER_ALBUM = 8           # Amount of reviews per album 
PAGES_PER_CHART = 10            # Amount of chart pages to scan for album
HUMAN_WAIT_SECS = 420
DELAY = (1.0, 2.0)

STAR_RE = re.compile(r"([\d.]+)\s*star", re.I)


# Browser Setup
def make_driver():
    opts = uc.ChromeOptions()
    opts.add_argument("--disable-blink-features=AutomationControlled")
    return uc.Chrome(options=opts)


# Cloudflare Handling
def is_human_check(driver):
    src = driver.page_source.lower()
    cues = ["confirm you are human","verify you are human","i'm not a robot","cf-challenge"]
    if any(c in src for c in cues):
        return True
    try:
        return bool(driver.find_elements(By.CSS_SELECTOR, "iframe[src*='recaptcha'], #challenge-stage"))
    except:
        return False

def wait_for_clear(driver, timeout=HUMAN_WAIT_SECS):
    start = time.time()
    while time.time() - start < timeout:
        if not is_human_check(driver):
            return True
        time.sleep(2)
    return False

def load_page(driver, url):
    driver.get(url)
    if is_human_check(driver):
        if not wait_for_clear(driver):
            return None
    time.sleep(1)
    return BeautifulSoup(driver.page_source, "lxml")


# Extracting Data
def extract_album_artist(soup):
    title = soup.select_one("meta[property='og:title']")
    if title and " - " in title["content"]:
        artist, album = [p.strip() for p in title["content"].split(" - ", 1)]
        return album, artist
    return "", ""

def extract_review(block):
    text = " ".join(el.get_text(" ", strip=True) for el in block.select("span.rendered_text, p")).strip()
    rating = ""
    tag = block.select_one("[aria-label*='star'], [title*='star']")
    if tag:
        m = STAR_RE.search(tag.get("aria-label","")+tag.get("title",""))
        if m: rating = m.group(1)
    return text, rating


# Scraping
def collect_album_links(driver):
    """Collect album URLs from the charts pages."""
    albums = set()
    for chart in CHARTS:
        for p in range(1, PAGES_PER_CHART+1):
            url = chart if p == 1 else f"{chart}?page={p}"
            soup = load_page(driver, url)
            if not soup: continue
            for a in soup.select("a[href*='/release/']"):
                href = a.get("href") or ""
                if "/release/" in href:
                    albums.add(urljoin(BASE, href))
            time.sleep(0.5)
    return sorted(albums)

def scrape_album(driver, url):
    """Scrape reviews from a single album (up to REVIEWS_PER_ALBUM)."""
    rows = []
    soup = load_page(driver, url + "/reviews/")
    if not soup: return rows

    album, artist = extract_album_artist(soup)

    while len(rows) < REVIEWS_PER_ALBUM and soup:
        blocks = soup.select("div.review, div.review_box, table.mbgen")
        if not blocks: break
        for block in blocks:
            text, rating = extract_review(block)
            if text or rating:
                rows.append({
                    "album": album,
                    "artist": artist,
                    "review_text": text,
                    "user_rating": rating
                })
            if len(rows) >= REVIEWS_PER_ALBUM:
                break
        nxt = soup.select_one("a[rel='next']")
        if nxt:
            soup = load_page(driver, urljoin(BASE, nxt.get("href")))
        else:
            break
        time.sleep(random.uniform(*DELAY))
    return rows


# Main Run
def run():
    driver = make_driver()
    all_reviews = []
    try:
        albums = collect_album_links(driver)
        for url in albums:
            all_reviews.extend(scrape_album(driver, url))
            pd.DataFrame(all_reviews).to_csv(OUT_FILE, index=False)
            time.sleep(random.uniform(*DELAY))
    finally:
        driver.quit()

if __name__ == "__main__":
    run()


In [3]:
import os
import re
import time
from datetime import datetime, timezone
from typing import List, Dict, Any

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import praw


# ----------------------------
# Config
# ----------------------------
SUBREDDIT = "LonghornNation"
QUERY = '"Arch Manning" OR Arch OR Manning'  # adjust if too broad
TIME_FILTER = "year"  # hour, day, week, month, year, all
POST_LIMIT = 200      # how many matching posts to pull
MAX_COMMENTS_PER_POST = 500  # cap for speed (None for all, but can be heavy)

OUT_POSTS_CSV = "arch_posts.csv"
OUT_COMMENTS_CSV = "arch_comments.csv"
OUT_SUMMARY_CSV = "arch_summary_by_day.csv"


# ----------------------------
# Helpers
# ----------------------------
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)
    # remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def utc_to_dt(utc_ts: float) -> str:
    return datetime.fromtimestamp(utc_ts, tz=timezone.utc).isoformat()


# ----------------------------
# Main
# ----------------------------
def main():
    load_dotenv()

    client_id = os.getenv("REDDIT_CLIENT_ID")
    client_secret = os.getenv("REDDIT_CLIENT_SECRET")
    user_agent = os.getenv("REDDIT_USER_AGENT")

    if not all([client_id, client_secret, user_agent]):
        raise ValueError("Missing Reddit API creds. Check your .env file.")

    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent,
    )

    analyzer = SentimentIntensityAnalyzer()

    posts_rows: List[Dict[str, Any]] = []
    comments_rows: List[Dict[str, Any]] = []

    subreddit = reddit.subreddit(SUBREDDIT)

    # Pull posts that match the search query
    # Note: Reddit search isn't perfect; if results are too broad, tighten QUERY.
    submissions = list(subreddit.search(QUERY, sort="new", time_filter=TIME_FILTER, limit=POST_LIMIT))

    for sub in tqdm(submissions, desc="Scraping posts"):
        posts_rows.append({
            "post_id": sub.id,
            "created_utc": sub.created_utc,
            "created_iso": utc_to_dt(sub.created_utc),
            "title": sub.title,
            "selftext": sub.selftext,
            "score": sub.score,
            "num_comments": sub.num_comments,
            "upvote_ratio": getattr(sub, "upvote_ratio", None),
            "author": str(sub.author) if sub.author else None,
            "permalink": f"https://www.reddit.com{sub.permalink}",
            "url": sub.url,
            "flair": getattr(sub, "link_flair_text", None),
        })

        # Combine title + body for post sentiment
        post_text = clean_text(f"{sub.title}\n{sub.selftext}")
        post_sent = analyzer.polarity_scores(post_text)
        posts_rows[-1].update({
            "sent_compound": post_sent["compound"],
            "sent_pos": post_sent["pos"],
            "sent_neu": post_sent["neu"],
            "sent_neg": post_sent["neg"],
        })

        # Comments: replace "MoreComments" so we can iterate
        try:
            sub.comments.replace_more(limit=0)
        except Exception:
            # If rate-limited or transient errors, pause briefly and continue
            time.sleep(2)
            try:
                sub.comments.replace_more(limit=0)
            except Exception:
                continue

        count = 0
        for c in sub.comments.list():
            if MAX_COMMENTS_PER_POST is not None and count >= MAX_COMMENTS_PER_POST:
                break
            if not getattr(c, "body", None):
                continue

            body = clean_text(c.body)
            sent = analyzer.polarity_scores(body)

            comments_rows.append({
                "post_id": sub.id,
                "comment_id": c.id,
                "created_utc": c.created_utc,
                "created_iso": utc_to_dt(c.created_utc),
                "author": str(c.author) if c.author else None,
                "score": c.score,
                "is_submitter": getattr(c, "is_submitter", None),
                "parent_id": c.parent_id,
                "permalink": f"https://www.reddit.com{c.permalink}",
                "body": body,
                "sent_compound": sent["compound"],
                "sent_pos": sent["pos"],
                "sent_neu": sent["neu"],
                "sent_neg": sent["neg"],
            })
            count += 1

    # Save datasets
    posts_df = pd.DataFrame(posts_rows)
    comments_df = pd.DataFrame(comments_rows)

    posts_df.to_csv(OUT_POSTS_CSV, index=False)
    comments_df.to_csv(OUT_COMMENTS_CSV, index=False)

    # Simple daily aggregation (comments tend to be richer signal than posts)
    if not comments_df.empty:
        comments_df["date"] = pd.to_datetime(comments_df["created_iso"]).dt.date
        summary = (comments_df
                   .groupby("date")
                   .agg(
                       n_comments=("comment_id", "count"),
                       avg_sent=("sent_compound", "mean"),
                       med_sent=("sent_compound", "median"),
                       avg_score=("score", "mean")
                   )
                   .reset_index()
                   .sort_values("date"))
        summary.to_csv(OUT_SUMMARY_CSV, index=False)

    print("Done.")
    print(f"- Posts: {OUT_POSTS_CSV} ({len(posts_df)})")
    print(f"- Comments: {OUT_COMMENTS_CSV} ({len(comments_df)})")
    if not comments_df.empty:
        print(f"- Summary: {OUT_SUMMARY_CSV} ({len(summary)})")


if __name__ == "__main__":
    main()


ValueError: Missing Reddit API creds. Check your .env file.